In [3]:
import math_toolkit
math_toolkit.activate()

# LpNorm

Write weighted symbolic Lebesgue norms with explicit variables, domains, and rewrites to their analytic definitions.

`L1Norm(expr, vars, domain, weight=1)`  
`L2Norm(expr, vars, domain, weight=1)`  
`LinftyNorm(expr, vars, domain, weight=1)`  
`LpNorm(p)(expr, vars, domain, weight=1)`


In [4]:
profile = L2Norm(sin(pi*x), x, (0, 1))

r"""
{{ profile }} = {{ profile.rewrite(Integral) >>DoIt }}
""" >> md


$\left\| \sin{\left(\pi x \right)} \right\|_{L^{2}(x)}$ = $\frac{\sqrt{2}}{2}$


## Core behavior

A norm object keeps the expression, bound variables, domain, exponent, and optional weight together as unevaluated notation. It stays symbolic until you ask for a rewrite or evaluation.


In [3]:
f = Function("f")
w = Function("w")
L = Symbol("L", positive=True)

N = L2Norm(f(x, t), x, (0, L), weight=w(x))
N

L2Norm(f(x, t), x, Interval(0, L), weight=w(x))

In [4]:
pretty("""The semantic fields remain inspectable.

- expression: {{ N.expr }}
- variables: {{ N.variables }}
- domain: {{ N.domain }}
- weight: {{ N.weight }}
- exponent: {{ N.p }}
""")

The semantic fields remain inspectable.

- expression: $f{\left(x,t \right)}$
- variables: (x,)
- domain: $\left[0, L\right]$
- weight: $w{\left(x \right)}$
- exponent: $2$


Finite norms rewrite to explicit `Integral` expressions. This makes the norm useful as compact notation while still letting SymPy integrate when it can.


In [5]:
N.rewrite(Integral)

       ______________________
      ╱ L                    
     ╱  ⌠                    
    ╱   ⎮               2    
   ╱    ⎮ w(x)⋅│f(x, t)│  dx 
  ╱     ⌡                    
╲╱      0                    

## Common patterns

**Named finite norms.** Use `L1Norm` and `L2Norm` directly when the exponent is part of the notation.


In [6]:
Tuple(
    L1Norm(f(x), x, (0, 1)),
    L2Norm(f(x), x, (0, 1)),
)

(L1Norm(f(x), x, Interval(0, 1)), L2Norm(f(x), x, Interval(0, 1)))

**Other finite exponents.** `LpNorm(p)` returns a norm constructor for the chosen exponent. The public shorthand classes are reused for `p = 1`, `p = 2`, and `p = oo`.


In [7]:
L3Norm = LpNorm(3)
Tuple(
    LpNorm(1) is L1Norm,
    LpNorm(2) is L2Norm,
    LpNorm(oo) is LinftyNorm,
    L3Norm(f(x), x, (0, 1)),
)

(True, True, True, _FiniteLpNorm_Integer_3_(f(x), x, Interval(0, 1)))

**Several variables.** Pass variables in the order you want them to appear in the rectangular product domain.


In [8]:
two_dimensional = L2Norm(f(x, y), (x, y), ((0, 1), (0, 2)))

two_dimensional.rewrite(Integral)

       ______________________
      ╱ 2 1                  
     ╱  ⌠ ⌠                  
    ╱   ⎮ ⎮          2       
   ╱    ⎮ ⎮ │f(x, y)│  dx dy 
  ╱     ⌡ ⌡                  
╲╱      0 0                  

**Weights.** The `weight=` argument is a nonnegative density with respect to Lebesgue measure. It appears in finite-norm rewrites as a factor in the integrand.


In [9]:
weighted = L2Norm(f(x), x, (0, 1), weight=1 + x**2)

weighted.rewrite(Integral)

       _______________________
      ╱ 1                     
     ╱  ⌠                     
    ╱   ⎮ ⎛ 2    ⎞       2    
   ╱    ⎮ ⎝x  + 1⎠⋅│f(x)│  dx 
  ╱     ⌡                     
╲╱      0                     

## Options and Details

**Domain forms.** A one-dimensional domain may be an endpoint pair, an `Interval`, or `S.Reals`. Several variables may use endpoint pairs or a rectangular `ProductSet`.


In [10]:
Tuple(
    L2Norm(f(x), x, (0, 1)).domain,
    L2Norm(f(x), x, Interval(0, 1)).domain,
    L2Norm(f(x), x, S.Reals).rewrite(Integral),
    L2Norm(f(x, y), (x, y), Interval(0, 1) * Interval(0, 2)).domain,
)

⎛                       _______________                 ⎞
⎜                      ╱ ∞                              ⎟
⎜                     ╱  ⌠                              ⎟
⎜                    ╱   ⎮        2                     ⎟
⎜[0, 1], [0, 1],    ╱    ⎮  │f(x)│  dx , [0, 1] × [0, 2]⎟
⎜                  ╱     ⌡                              ⎟
⎝                ╲╱      -∞                             ⎠

## Numerical evaluation

Norm expressions can cross the numerical boundary with `>> Num`. Finite `Lp` norms are expanded to their integral representation and then evaluated by the same numerical integration machinery used for explicit `Integral` objects.


In [9]:
_=L2Norm(sin(x), x, (0, pi)) >> Num 

r""""
{{ L2Norm(sin(x), x, (0, pi)) }}={{ _ }}
""" >>md

_ = L1Norm(x, x, (0, 1)) >> Num
r""""
{{ L1Norm(x, x, (0, 1)) }}={{ _ }}
""" >>md


"
$\left\| \sin{\left(x \right)} \right\|_{L^{2}(x)}$=1.2533141373155001


"
$\left\| x \right\|_{L^{1}(x)}$=0.5


Free symbols become numerical parameters. Use `>> Num(var=a)` to make the parameter order explicit; array inputs are broadcast by NumPy.


In [11]:
parameterized_l2 = L2Norm(a*x, x, (0, 1)) >> Num(var=a)

parameterized_l2(np.array([-2.0, 0.0, 3.0]))

array([1.15470054, 0.        , 1.73205081])

`LinftyNorm` uses a custom numerical lowering. It maximizes the absolute value over a bounded interval or rectangular product domain.


In [ ]:
LinftyNorm(x*(1 - x), x, (0, 1)) >> Num

In [ ]:
parameterized_linf = LinftyNorm(a*sin(x), x, (0, pi)) >> Num(var=a)

parameterized_linf(np.array([-2.0, 0.5, 3.0]))

Numerical infinity norms are intentionally conservative. They require bounded interval or rectangular product domains, and weighted infinity norms are not numerically supported yet. Finite norms can still use unbounded domains when the underlying numerical integral can handle them.


In [ ]:
from math_toolkit.num import NumUnsupportedExpressionError

try:
    LinftyNorm(exp(-x**2), x, S.Reals) >> Num
except NumUnsupportedExpressionError as err:
    str(err)

**Display.** Plain text includes the domain and weight for inspection. LaTeX uses compact norm notation and keeps those details hidden unless you inspect the object fields or rewrite it.


In [12]:
compact = L2Norm(f(x), x, (0, 1), weight=1 + x)

(
    r"""- Plain text: `{{ sstr(compact) }}`
- LaTeX: ${{ latex(compact) }}$
"""
    >> md
)

- Plain text: `L2Norm(f(x), x, Interval(0, 1), weight=x + 1)`
- LaTeX: $\left\| f{\left(x \right)} \right\|_{L^{2}(x)}$


## Semantics

The variables listed in `vars` are binders. They are local to the norm, so they are not free symbols even when they appear in the expression or weight.


In [14]:
w=Function("w")
bound_example = L2Norm(f(x, t), x, (0, L), weight=w(x, a))

"""- bound symbols: {{ bound_example.bound_symbols }}
- free symbols: {{ bound_example.free_symbols }}
""" >> md

- bound symbols: [x]
- free symbols: {a, L, t}


Substitution respects that binding. Replacing the bound variable does not change the norm, while replacing a free parameter does.


In [13]:
Tuple(
    bound_example.subs(x, y),
    bound_example.subs(t, s),
    bound_example.subs(L, 2),
)

(L2Norm(f(x, t), x, Interval(0, L), weight=w(x, a)), L2Norm(f(x, s), x, Interv ↪

↪ al(0, L), weight=w(x, a)), L2Norm(f(x, t), x, Interval(0, 2), weight=w(x, a) ↪

↪ ))

Use `xreplace` when you deliberately want structural renaming, including renaming the bound variable itself.


In [14]:
L2Norm(f(x, t), x, (0, 1)).xreplace({x: y})

L2Norm(f(y, t), y, Interval(0, 1))

`LinftyNorm` represents an essential supremum, not an ordinary maximum. Ask for the explicit formal object with `rewrite(EssentialSupremum)`.


In [15]:
LinftyNorm(f(x), x, (0, 1)).rewrite(EssentialSupremum)

EssentialSupremum(Abs(f(x)), x, Interval(0, 1))

## Examples and Applications

### Example: piecewise functions

Norms keep `Piecewise` expressions symbolic until the rewrite and evaluation step. When SymPy can integrate the resulting expression, `.doit()` gives the corresponding value.


In [16]:
piecewise_profile = If(x < 0).Then(x).Otherwise(2*x)

Tuple(
    L1Norm(piecewise_profile, x, (-1, 1)).rewrite(Integral).doit(),
    L2Norm(piecewise_profile, x, (-1, 1)).rewrite(Integral).doit(),
)

⎛     √15⎞
⎜3/2, ───⎟
⎝      3 ⎠

Weights may be piecewise too. The weight is part of the measure density, so it multiplies the finite-norm integrand.


In [17]:
piecewise_weight = If(x < 0).Then(1).Otherwise(3)

L1Norm(1, x, (-1, 1), weight=piecewise_weight).rewrite(Integral).doit()

4

### Application: compact energy notation

Norm notation can keep a variational expression readable while still allowing the analytic form to be expanded when needed.


In [18]:
u = Function("u")
energy = Rational(1,2) * L2Norm(diff(u(x), x), x, (0, 1))**2
energy

                                              2
L2Norm(Derivative(u(x), x), x, Interval(0, 1)) 
───────────────────────────────────────────────
                       2                       

In [19]:
energy.rewrite(Integral)

1               
⌠               
⎮           2   
⎮ │d       │    
⎮ │──(u(x))│  dx
⎮ │dx      │    
⌡               
0               
────────────────
       2        

## Pitfalls

**Variables and domains are explicit.** `L2Norm(f(x))` is not enough information, because the same expression might be normed in `x`, in another variable, or over a different region.

**Bound variables cannot define their own endpoints.** Use a different parameter name for external limits.


In [20]:
try:
    L2Norm(f(x), x, (x, 1))
except ValueError as err:
    str(err)

**Unsupported domains may be stored but not integrated.** A symbolic domain can be useful notation, but rewriting to `Integral` currently supports intervals and rectangular products.


In [21]:
Omega = Symbol("Omega")
symbolic_domain_norm = L2Norm(f(x, y), (x, y), Omega)

try:
    symbolic_domain_norm.rewrite(Integral)
except NotImplementedError as err:
    pretty("""The symbolic domain is stored as {{ symbolic_domain_norm.domain }}, but rewrite reports: `{{ str(err) }}`.""")

The symbolic domain is stored as $\Omega$, but rewrite reports: `only interval and rectangular product domains are supported`.

## See also

[Expression](Expression.ipynb)  
[Function](Function.ipynb)  
[rewrite](rewrite.ipynb)  
[subs](subs.ipynb)  
[If](If.ipynb)  
[Composing structured expressions](../tutorials/composing_expressions.ipynb)
